In [1]:
import os

# Ensure protobuf works with TensorFlow
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
import pandas as pd
import numpy as np
import tensorflow as tf
import glob
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

SEED = 42
DEBUG = False
tf.random.set_seed(SEED)




2026-05-02 13:50:45.734116: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 13:50:45.770115: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777729845.792294      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777729845.801610      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 13:50:45.841333: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Build EfficientNetB4 backbone with ImageNet weights and a classification head
base_model = EfficientNetB4(
    include_top=False, weights="imagenet", input_shape=(300, 300, 3)
)
x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(5, activation="softmax")(x)
my_model = Model(inputs=base_model.input, outputs=output)

my_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)




2026-05-02 13:51:09.059981: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [3]:
# Paths to training CSV and images
train_csv_path = "../input/cassava-leaf-disease-classification/train.csv"
train_images_dir = "../input/cassava-leaf-disease-classification/train_images/"

df_train = pd.read_csv(train_csv_path)
df_train["path"] = df_train["image_id"].apply(
    lambda x: os.path.join(train_images_dir, x)
)

# Keep labels as integer dtype for sparse loss
df_train["label"] = df_train["label"].astype(int)

# Train‑validation split
train_df, val_df = train_test_split(
    df_train,
    test_size=0.2,
    random_state=SEED,
    stratify=df_train["label"],
)

# Data augmentation
train_idg = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)
val_idg = ImageDataGenerator(rescale=1.0 / 255)

# Generators – use sparse class mode (integer labels)
train_gen = train_idg.flow_from_dataframe(
    dataframe=train_df,
    x_col="path",
    y_col="label",
    target_size=(300, 300),
    batch_size=32,
    class_mode="sparse",  # returns integer labels compatible with loss
    seed=SEED,
    shuffle=True,
)

val_gen = val_idg.flow_from_dataframe(
    dataframe=val_df,
    x_col="path",
    y_col="label",
    target_size=(300, 300),
    batch_size=32,
    class_mode="sparse",
    seed=SEED,
    shuffle=False,
)

# Train the model (more epochs to raise accuracy)
my_model.fit(train_gen, epochs=15, validation_data=val_gen, verbose=1)




TypeError: If class_mode="sparse", y_col="label" column values must be strings.

In [4]:
# Prepare test data generator
test_images = glob.glob(
    "../input/cassava-leaf-disease-classification/test_images/*.jpg"
)
df_test = pd.DataFrame(test_images, columns=["path"])


def make_test_gen(batch_size=128):
    test_idg = ImageDataGenerator(rescale=1.0 / 255)
    test_gen = test_idg.flow_from_dataframe(
        dataframe=df_test,
        x_col="path",
        y_col=None,
        batch_size=batch_size,
        seed=SEED,
        shuffle=False,
        class_mode=None,
        target_size=(300, 300),
    )
    return test_gen


test_gen = make_test_gen()
pred_test = my_model.predict(test_gen, verbose=1)
pred_test_labels = np.argmax(pred_test, axis=-1)




Found 2676 validated image filenames.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


21/21 ━━━━━━━━━━━━━━━━━━━━ 248s 11s/step


In [5]:
# Build submission file matching required format
final_submission = df_test.copy()
final_submission["image_id"] = final_submission.path.apply(
    lambda p: os.path.basename(p)
)
final_submission["label"] = pred_test_labels.astype(int)

final_csv = final_submission[["image_id", "label"]]
final_csv.to_csv("submission.csv", index=False)